# Lab2.2 — Acces la AXI GPIO din Python

În Lab1 am folosit interfața de nivel înalt `base.leds`, `base.buttons` și `base.switches`. Acum accesăm direct controlerele **AXI GPIO** din Programmable Logic.


## Obiective

- identificarea unui AXI GPIO din `ip_dict`;
- utilizarea clasei `AxiGPIO`;
- configurarea direcției unui canal GPIO;
- citirea și scrierea valorilor pe mai mulți biți;
- înțelegerea nivelului de abstractizare oferit de driverul PYNQ.


## 1. Inițializare


In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq.lib import AxiGPIO
import time

base = BaseOverlay("base.bit")

Identificăm controlerele GPIO după numele lor din Overlay.


In [ ]:
def find_ip_name(words, exclude=()):
    for name, info in base.ip_dict.items():
        text = (name + " " + info.get("type", "")).lower()
        if "gpio" in text and any(w in name.lower() for w in words):
            if not any(x in name.lower() for x in exclude):
                return name
    raise RuntimeError(f"GPIO IP not found for {words}")

led_name = find_ip_name(("led",), exclude=("rgb",))
button_name = find_ip_name(("button", "btn"))
switch_name = find_ip_name(("switch", "sw"))

print("LED GPIO    :", led_name)
print("Button GPIO :", button_name)
print("Switch GPIO :", switch_name)

## 2. Controlul LED-urilor prin `AxiGPIO`

Creăm un driver `AxiGPIO` pornind de la descrierea IP-ului furnizată de `ip_dict`.


In [ ]:
leds = AxiGPIO(base.ip_dict[led_name]).channel1
leds.setdirection("out")
leds.setlength(4)

print("LED AXI GPIO initialized.")

In [ ]:
for value in (0b0001, 0b0010, 0b0100, 0b1000, 0b1111, 0b0000):
    leds.write(value, 0b1111)
    print(f"LED value = {value:04b}")
    time.sleep(0.5)

## 3. Citirea butoanelor și switch-urilor


In [ ]:
buttons = AxiGPIO(base.ip_dict[button_name]).channel1
buttons.setdirection("in")
buttons.setlength(4)

switches = AxiGPIO(base.ip_dict[switch_name]).channel1
switches.setdirection("in")
switches.setlength(2)

print(f"Buttons  = {buttons.read() & 0xF:04b}")
print(f"Switches = {switches.read() & 0x3:02b}")

## 4. Aplicație: butoane → LED-uri

Cât timp SW0 este `1`, starea celor patru butoane este copiată pe cele patru LED-uri. Pentru oprire, comutați SW0 pe `0`.


In [ ]:
while (switches.read() & 0x1) == 1:
    leds.write(buttons.read() & 0xF, 0b1111)
    time.sleep(0.01)

leds.write(0, 0b1111)
print("Program oprit.")

## 5. Niveluri de abstractizare

În Lab1 am folosit:

```python
base.leds[0].on()
```

În acest task folosim direct driverul AXI GPIO:

```python
leds.write(0b0001, 0b1111)
```

Ambele variante ajung la același periferic hardware, dar la niveluri diferite de abstractizare.


## 6. Exercițiu

Scrieți un program în care valoarea celor două switch-uri, interpretată ca număr 0...3, selectează unul dintre cele patru LED-uri. Implementați soluția folosind **numai obiectul `AxiGPIO`**, fără `base.leds` sau `base.switches`.


In [ ]:
# Scrieți aici soluția


## 7. Întrebări

1. Ce avantaj oferă clasa `AxiGPIO` față de accesul direct la registre?
2. De ce LED-urile sunt configurate ca ieșiri, iar butoanele ca intrări?
3. Ce rol are masca `0b1111` din apelul `write()`?
